# 03 — Silver Transformations

Typed, deduplicated tables built from bronze VARIANT payloads.

- `data:field::type` for scalar paths
- `from_json(data:field::string, '<schema>')` + `LATERAL VIEW EXPLODE` for array paths.
  Each silver block carries its own inline DDL string so it's self-contained.
  See `sportlogiq_schemas.py` for the full PySpark `StructType` definitions of every
  route response — useful as **reference documentation** when authoring new silver blocks.
- **MD5 surrogate keys** so silver is restart-safe
- **`INSERT OVERWRITE`** for full-refresh idempotency (MERGE isn't supported through
  Databricks Connect; in-workspace runs can swap to MERGE if you prefer incremental)
- **PK / FK RELY** constraints + **liquid clustering** on game-date / team / player

| Silver table | Grain | Clustered by |
|--------------|-------|--------------|
| `silver_competitions` | competition | — |
| `silver_teams` | team | — |
| `silver_team_records` | team × season-to-date | — |
| `silver_venues` | venue | — |
| `silver_players` | player | — |
| `silver_metric_topics` | (scope, topic_id) | — |
| `silver_games` | game | `(game_date, home_team_id)` |
| `silver_game_rosters` | game × person | — |
| `silver_compiled_events` | event | `(game_id, period)` |
| `silver_full_events` | event | `(game_id, period)` |
| `silver_shift_events` | shift | `(game_id, player_id)` |
| `silver_player_toi` | game × player × period | — |
| `silver_player_game_metrics` | game × player × metric | `(game_id, player_id)` |
| `silver_season_metrics` | scope × player/team × metric | `(scope, season)` |

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()

# Dual-mode Spark session — works locally via Databricks Connect AND inside a
# Databricks workspace notebook. In the workspace, `spark` already exists.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [ ]:
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "sportlogiq_nhl")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Catalog: {UC_CATALOG}")
print(f"Bronze:  {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver:  {UC_CATALOG}.{SILVER_SCHEMA}")
print(f"Gold:    {UC_CATALOG}.{GOLD_SCHEMA}")
print(f"Volume:  {VOLUME_PATH}")

In [ ]:
B = f"{UC_CATALOG}.{BRONZE_SCHEMA}"
S = f"{UC_CATALOG}.{SILVER_SCHEMA}"
print(f"Bronze: {B}\nSilver: {S}")

In [ ]:
def add_pk(table: str, name: str, cols: list):
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT 1 FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'PRIMARY KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        return
    for c in cols:
        try:
            spark.sql(f"ALTER TABLE {table} ALTER COLUMN {c} SET NOT NULL")
        except Exception as e:
            if "ALREADY_NOT_NULLABLE" not in str(e):
                raise
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} PRIMARY KEY ({', '.join(cols)}) RELY")
    print(f"  + PK {name} on {table}")


def add_fk(table: str, name: str, cols: list, ref_table: str, ref_cols: list):
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT 1 FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'FOREIGN KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        return
    spark.sql(
        f"ALTER TABLE {table} ADD CONSTRAINT {name} "
        f"FOREIGN KEY ({', '.join(cols)}) REFERENCES {ref_table} ({', '.join(ref_cols)}) RELY"
    )
    print(f"  + FK {name} on {table}")


## Reference dimensions

In [ ]:
# Competitions  (one row per league SportLogiq has data for)
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_competitions AS
    SELECT
        data:id::string                 AS competition_id,
        data:name::string               AS competition_name,
        data:seasons::string            AS seasons_raw,
        _ingestion_timestamp
    FROM {B}.bronze_competitions
""")
add_pk(f"{S}.silver_competitions", "silver_competitions_pk", ["competition_id"])
print("silver_competitions:", spark.table(f"{S}.silver_competitions").count())

In [ ]:
# Teams (explode the embedded `teams` array — one row per call from SportLogiq)
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_teams AS
    WITH exploded AS (
        SELECT data:competition_id::int AS competition_id,
               data:season::string      AS season,
               data:stage::string       AS stage,
               team,
               _ingestion_timestamp
        FROM {B}.bronze_teams
        LATERAL VIEW EXPLODE(
            FROM_JSON(data:teams::string,
                      'ARRAY<STRUCT<id:STRING,name:STRING,location:STRING,shorthand:STRING,division:STRING,conference:STRING,logo_src:STRING>>')
        ) AS team
    )
    SELECT
        team.id::int                AS team_id,
        team.name                   AS team_name,
        team.location               AS team_location,
        team.shorthand              AS team_shorthand,
        team.division               AS division,
        team.conference             AS conference,
        team.logo_src               AS logo_src,
        competition_id,
        season,
        stage,
        _ingestion_timestamp
    FROM exploded
""")
add_pk(f"{S}.silver_teams", "silver_teams_pk", ["team_id"])
print("silver_teams:", spark.table(f"{S}.silver_teams").count())

In [ ]:
# Team records — flattens the records array into one row per (team, season)
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_team_records AS
    WITH exploded AS (
        SELECT rec, _ingestion_timestamp
        FROM {B}.bronze_team_records
        LATERAL VIEW EXPLODE(
            FROM_JSON(data:teams::string,
                      'ARRAY<STRUCT<id:STRING,record:STRUCT<wins:INT,losses:INT,overtime_losses:INT,ties:INT>>>')
        ) AS rec
    )
    SELECT
        rec.id::int                            AS team_id,
        COALESCE(rec.record.wins, 0)           AS wins,
        COALESCE(rec.record.losses, 0)         AS losses,
        COALESCE(rec.record.overtime_losses,0) AS overtime_losses,
        COALESCE(rec.record.ties, 0)           AS ties,
        (COALESCE(rec.record.wins,0) * 2 + COALESCE(rec.record.overtime_losses,0)
         + COALESCE(rec.record.ties,0))        AS standings_points,
        _ingestion_timestamp
    FROM exploded
""")
add_pk(f"{S}.silver_team_records", "silver_team_records_pk", ["team_id"])
add_fk(f"{S}.silver_team_records", "silver_team_records_team_fk",
       ["team_id"], f"{S}.silver_teams", ["team_id"])
print("silver_team_records:", spark.table(f"{S}.silver_team_records").count())

In [ ]:
# Venues
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_venues AS
    WITH exploded AS (
        SELECT v, _ingestion_timestamp
        FROM {B}.bronze_venues
        LATERAL VIEW EXPLODE(
            FROM_JSON(data:venues::string,
                      'ARRAY<STRUCT<id:STRING,name:STRING,city:STRING,state:STRING,country:STRING,timezone:STRING,capacity:INT>>')
        ) AS v
    )
    SELECT
        v.id::int      AS venue_id,
        v.name         AS venue_name,
        v.city         AS city,
        v.state        AS state,
        v.country      AS country,
        v.timezone     AS timezone,
        v.capacity     AS capacity,
        _ingestion_timestamp
    FROM exploded
""")
add_pk(f"{S}.silver_venues", "silver_venues_pk", ["venue_id"])
print("silver_venues:", spark.table(f"{S}.silver_venues").count())

In [ ]:
# Players (league-wide, latest pull only — keeps the most recent ingestion timestamp per player)
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_players AS
    WITH exploded AS (
        SELECT data:season::string AS season,
               p,
               _ingestion_timestamp
        FROM {B}.bronze_players
        LATERAL VIEW EXPLODE(
            FROM_JSON(data:players::string,
                      'ARRAY<STRUCT<id:STRING,first_name:STRING,last_name:STRING,current_team_id:STRING,position:STRING,role:STRING,status:STRING>>')
        ) AS p
    ),
    deduped AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY p.id ORDER BY _ingestion_timestamp DESC) AS rn
        FROM exploded
    )
    SELECT
        p.id::int                AS player_id,
        p.first_name             AS first_name,
        p.last_name              AS last_name,
        p.current_team_id::int   AS current_team_id,
        p.position               AS position,
        p.role                   AS role,
        p.status                 AS status,
        season,
        _ingestion_timestamp
    FROM deduped
    WHERE rn = 1
""")
add_pk(f"{S}.silver_players", "silver_players_pk", ["player_id"])
print("silver_players:", spark.table(f"{S}.silver_players").count())

In [ ]:
# Metric topics catalog (per scope)
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_metric_topics AS
    WITH exploded AS (
        SELECT data:scope::string AS scope, t, _ingestion_timestamp
        FROM {B}.bronze_metric_topics
        LATERAL VIEW EXPLODE(
            FROM_JSON(data:topics::string, 'ARRAY<STRUCT<id:STRING,name:STRING>>')
        ) AS t
    )
    SELECT
        scope,
        t.id        AS topic_id,
        t.name      AS topic_name,
        _ingestion_timestamp
    FROM exploded
""")
add_pk(f"{S}.silver_metric_topics", "silver_metric_topics_pk", ["scope", "topic_id"])
print("silver_metric_topics:", spark.table(f"{S}.silver_metric_topics").count())

## Game-grain facts

In [ ]:
# Games
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_games (
        game_id          INT NOT NULL,
        season           STRING,
        stage            STRING,
        game_date        DATE,
        scheduled_time   STRING,
        venue_id         INT,
        home_team_id     INT,
        away_team_id     INT,
        winning_team_id  INT,
        home_score_array STRING,
        away_score_array STRING,
        _ingestion_timestamp TIMESTAMP
    )
    CLUSTER BY (game_date, home_team_id)
""")
spark.sql(f"""
    INSERT OVERWRITE {S}.silver_games
    SELECT
        data:id::int                       AS game_id,
        data:season::string                AS season,
        data:stage::string                 AS stage,
        TRY_CAST(data:date::string AS DATE) AS game_date,
        data:scheduled_time::string        AS scheduled_time,
        data:venue_id::int                 AS venue_id,
        data:home_team.id::int             AS home_team_id,
        data:away_team.id::int             AS away_team_id,
        data:winning_team_id::int          AS winning_team_id,
        data:home_score::string            AS home_score_array,
        data:away_score::string            AS away_score_array,
        _ingestion_timestamp
    FROM {B}.bronze_game
    WHERE data:id IS NOT NULL
""")
add_pk(f"{S}.silver_games", "silver_games_pk", ["game_id"])
add_fk(f"{S}.silver_games", "silver_games_home_fk",
       ["home_team_id"], f"{S}.silver_teams", ["team_id"])
add_fk(f"{S}.silver_games", "silver_games_away_fk",
       ["away_team_id"], f"{S}.silver_teams", ["team_id"])
add_fk(f"{S}.silver_games", "silver_games_venue_fk",
       ["venue_id"], f"{S}.silver_venues", ["venue_id"])
print("silver_games:", spark.table(f"{S}.silver_games").count())

In [ ]:
# Game rosters — explode the crew array
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_game_rosters AS
    WITH exploded AS (
        SELECT
            CAST(REGEXP_EXTRACT(_source_file, 'games/([0-9]+)/roster.json', 1) AS INT) AS game_id,
            crew,
            _ingestion_timestamp
        FROM {B}.bronze_game_rosters
        LATERAL VIEW EXPLODE(
            FROM_JSON(data:crew::string,
                      'ARRAY<STRUCT<id:STRING,first_name:STRING,last_name:STRING,position:STRING,task:STRING,role:STRING>>')
        ) AS crew
    )
    SELECT
        MD5(CONCAT_WS('||', CAST(game_id AS STRING), crew.id))  AS roster_sk,
        game_id,
        crew.id::int       AS person_id,
        crew.first_name    AS first_name,
        crew.last_name     AS last_name,
        crew.position      AS position,
        crew.task          AS task,
        crew.role          AS role,
        _ingestion_timestamp
    FROM exploded
""")
add_pk(f"{S}.silver_game_rosters", "silver_game_rosters_pk", ["roster_sk"])
add_fk(f"{S}.silver_game_rosters", "silver_game_rosters_game_fk",
       ["game_id"], f"{S}.silver_games", ["game_id"])
print("silver_game_rosters:", spark.table(f"{S}.silver_game_rosters").count())

## Event-grain facts (compiled + full)

In [ ]:
# Compiled events — used for shot maps, shot attempts, possession analytics
EVENTS_COLS = """
    data:event_id::string                 AS event_id,
    data:game_reference_id::string        AS game_reference_id,
    data:x_coord::int                     AS x_coord,
    data:y_coord::int                     AS y_coord,
    data:zone::string                     AS zone,
    data:timecode::string                 AS timecode,
    data:frame::int                       AS frame,
    data:period::int                      AS period,
    data:period_time::int                 AS period_time,
    data:game_time::int                   AS game_time,
    data:minutes::int                     AS minutes,
    data:seconds::int                     AS seconds,
    data:team::string                     AS team,
    data:player_id::int                   AS player_id,
    data:player_position::string          AS player_position,
    data:player_first_name::string        AS player_first_name,
    data:player_last_name::string         AS player_last_name,
    data:player_jersey::string            AS player_jersey,
    data:event_sequence_id::string        AS event_sequence_id,
    data:shorthand::string                AS shorthand,
    data:name::string                     AS name,
    data:type::string                     AS type,
    data:outcome::string                  AS outcome,
    data:flags::string                    AS flags_raw,
    data:players_involved_in_play::string AS players_involved_in_play,
    data:opposing_team_players_involved_in_play::string AS opposing_team_players_involved_in_play,
    data:current_possession::int          AS current_possession,
    data:team_in_possession::string       AS team_in_possession,
    data:current_play_in_possession::int  AS current_play_in_possession,
    data:current_off_play_in_possession::int AS current_off_play_in_possession,
    data:current_def_play_in_possession::int AS current_def_play_in_possession,
    data:related_event_id::string         AS related_event_id,
    data:related_event_player_id::int     AS related_event_player_id,
    data:attributes::string               AS attributes_raw
"""

for kind in ("compiled_events", "full_events"):
    target = f"{S}.silver_{kind}"
    spark.sql(f"""
        CREATE OR REPLACE TABLE {target} (
            base_event_id    STRING NOT NULL,
            game_id          INT,
            event_id         STRING,
            game_reference_id STRING,
            x_coord          INT,
            y_coord          INT,
            zone             STRING,
            timecode         STRING,
            frame            INT,
            period           INT,
            period_time      INT,
            game_time        INT,
            minutes          INT,
            seconds          INT,
            team             STRING,
            player_id        INT,
            player_position  STRING,
            player_first_name STRING,
            player_last_name STRING,
            player_jersey    STRING,
            event_sequence_id STRING,
            shorthand        STRING,
            name             STRING,
            type             STRING,
            outcome          STRING,
            flags_raw        STRING,
            players_involved_in_play STRING,
            opposing_team_players_involved_in_play STRING,
            current_possession INT,
            team_in_possession STRING,
            current_play_in_possession INT,
            current_off_play_in_possession INT,
            current_def_play_in_possession INT,
            related_event_id STRING,
            related_event_player_id INT,
            attributes_raw  STRING,
            _ingestion_timestamp TIMESTAMP
        )
        CLUSTER BY (game_id, period)
    """)
    spark.sql(f"""
        INSERT OVERWRITE {target}
        SELECT
            MD5(CONCAT_WS('||',
                COALESCE(CAST(REGEXP_EXTRACT(_source_file, 'games/([0-9]+)/', 1) AS STRING), ''),
                COALESCE(data:event_id::string, ''),
                COALESCE(data:event_sequence_id::string, '')
            )) AS base_event_id,
            CAST(REGEXP_EXTRACT(_source_file, 'games/([0-9]+)/', 1) AS INT) AS game_id,
            {EVENTS_COLS},
            _ingestion_timestamp
        FROM {B}.bronze_{kind}
    """)
    add_pk(target, f"silver_{kind}_pk", ["base_event_id"])
    add_fk(target, f"silver_{kind}_game_fk",
           ["game_id"], f"{S}.silver_games", ["game_id"])
    print(f"{target}: {spark.table(target).count():,}")

In [ ]:
# Shift events — same event-row schema as compiled, just a different bronze source
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_shift_events (
        shift_event_sk  STRING NOT NULL,
        game_id         INT,
        event_id        STRING,
        period          INT,
        period_time     INT,
        team            STRING,
        player_id       INT,
        type            STRING,
        outcome         STRING,
        x_coord         INT,
        y_coord         INT,
        zone            STRING,
        _ingestion_timestamp TIMESTAMP
    )
    CLUSTER BY (game_id, player_id)
""")
spark.sql(f"""
    INSERT OVERWRITE {S}.silver_shift_events
    SELECT
        MD5(CONCAT_WS('||',
            COALESCE(CAST(REGEXP_EXTRACT(_source_file, 'games/([0-9]+)/', 1) AS STRING), ''),
            COALESCE(data:event_id::string, ''),
            COALESCE(data:event_sequence_id::string, '')
        )) AS shift_event_sk,
        CAST(REGEXP_EXTRACT(_source_file, 'games/([0-9]+)/', 1) AS INT) AS game_id,
        data:event_id::string  AS event_id,
        data:period::int       AS period,
        data:period_time::int  AS period_time,
        data:team::string      AS team,
        data:player_id::int    AS player_id,
        data:type::string      AS type,
        data:outcome::string   AS outcome,
        data:x_coord::int      AS x_coord,
        data:y_coord::int      AS y_coord,
        data:zone::string      AS zone,
        _ingestion_timestamp
    FROM {B}.bronze_shift_events
""")
add_pk(f"{S}.silver_shift_events", "silver_shift_events_pk", ["shift_event_sk"])
print("silver_shift_events:", spark.table(f"{S}.silver_shift_events").count())

In [ ]:
# Player TOI — flattens the time-on-ice array (one row per game × player × period × strength state)
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_player_toi AS
    WITH exploded AS (
        SELECT
            CAST(REGEXP_EXTRACT(_source_file, 'games/([0-9]+)/', 1) AS INT) AS game_id,
            entry,
            _ingestion_timestamp
        FROM {B}.bronze_player_toi
        LATERAL VIEW OUTER EXPLODE(
            FROM_JSON(data::string,
                      'ARRAY<STRUCT<player_id:STRING,team:STRING,period:INT,strength:STRING,toi_seconds:INT>>')
        ) AS entry
    )
    SELECT
        MD5(CONCAT_WS('||',
            CAST(game_id AS STRING),
            COALESCE(entry.player_id, ''),
            CAST(COALESCE(entry.period,-1) AS STRING),
            COALESCE(entry.strength, '')
        )) AS toi_sk,
        game_id,
        entry.player_id::int AS player_id,
        entry.team           AS team,
        entry.period         AS period,
        entry.strength       AS strength,
        entry.toi_seconds    AS toi_seconds,
        _ingestion_timestamp
    FROM exploded
    WHERE entry IS NOT NULL
""")
add_pk(f"{S}.silver_player_toi", "silver_player_toi_pk", ["toi_sk"])
print("silver_player_toi:", spark.table(f"{S}.silver_player_toi").count())

## Per-game / per-season metric grids

In [ ]:
# Per-game player metrics — 1 row per (game_id, topic_id, breakdown_dim, metric_key)
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_player_game_metrics AS
    WITH base AS (
        SELECT
            CAST(REGEXP_EXTRACT(_source_file, 'games/([0-9]+)/', 1) AS INT) AS game_id,
            REGEXP_EXTRACT(_source_file, 'metrics_player_([^.]+).json', 1) AS topic_id,
            data,
            _ingestion_timestamp
        FROM {B}.bronze_game_metrics
    ),
    exploded AS (
        SELECT
            game_id, topic_id, _ingestion_timestamp,
            grouping
        FROM base
        LATERAL VIEW EXPLODE(
            FROM_JSON(data:groupings::string,
                      'ARRAY<STRUCT<breakdown:ARRAY<STRUCT<key:STRING,value:STRING>>,'
                      'metrics:ARRAY<STRUCT<key:STRING,value:DOUBLE>>,'
                      'unitsplayed:ARRAY<STRUCT<name:STRING,value:DOUBLE>>>>')
        ) AS grouping
    ),
    metric_rows AS (
        SELECT
            game_id, topic_id, _ingestion_timestamp,
            grouping.breakdown AS breakdown,
            metric.key   AS metric_key,
            metric.value AS metric_value
        FROM exploded
        LATERAL VIEW EXPLODE(grouping.metrics) AS metric
    )
    SELECT
        MD5(CONCAT_WS('||',
            CAST(game_id AS STRING),
            topic_id,
            CONCAT_WS(':', TRANSFORM(breakdown, x -> CONCAT(x.key, '=', x.value))),
            metric_key)) AS metric_sk,
        game_id,
        topic_id,
        CAST(FILTER(breakdown, x -> x.key = 'player')[0].value   AS INT) AS player_id,
        CAST(FILTER(breakdown, x -> x.key = 'team')[0].value     AS INT) AS team_id,
        CAST(FILTER(breakdown, x -> x.key = 'period')[0].value   AS INT) AS period,
        FILTER(breakdown, x -> x.key = 'position')[0].value      AS position,
        metric_key,
        metric_value,
        _ingestion_timestamp
    FROM metric_rows
""")
add_pk(f"{S}.silver_player_game_metrics", "silver_player_game_metrics_pk", ["metric_sk"])
print("silver_player_game_metrics:", spark.table(f"{S}.silver_player_game_metrics").count())

In [ ]:
# Season-level metric grid — same shape but keyed by (scope, topic, breakdown)
spark.sql(f"""
    CREATE OR REPLACE TABLE {S}.silver_season_metrics AS
    WITH base AS (
        SELECT
            REGEXP_EXTRACT(_source_file, 'season_metrics/([^/]+)/', 1) AS scope,
            REGEXP_EXTRACT(_source_file, 'season_metrics/[^/]+/([^.]+).json', 1) AS topic_id,
            data,
            _ingestion_timestamp
        FROM {B}.bronze_season_metrics
    ),
    exploded AS (
        SELECT
            scope, topic_id, data, _ingestion_timestamp, grouping
        FROM base
        LATERAL VIEW EXPLODE(
            FROM_JSON(data:groupings::string,
                      'ARRAY<STRUCT<breakdown:ARRAY<STRUCT<key:STRING,value:STRING>>,'
                      'metrics:ARRAY<STRUCT<key:STRING,value:DOUBLE>>,'
                      'unitsplayed:ARRAY<STRUCT<name:STRING,value:DOUBLE>>>>')
        ) AS grouping
    ),
    metric_rows AS (
        SELECT
            scope, topic_id, _ingestion_timestamp,
            grouping.breakdown AS breakdown,
            data:season::string AS season,
            metric.key   AS metric_key,
            metric.value AS metric_value
        FROM exploded
        LATERAL VIEW EXPLODE(grouping.metrics) AS metric
    )
    SELECT
        MD5(CONCAT_WS('||', scope, topic_id, season,
                            CONCAT_WS(':', TRANSFORM(breakdown, x -> CONCAT(x.key, '=', x.value))),
                            metric_key)) AS season_metric_sk,
        scope,
        topic_id,
        season,
        CAST(FILTER(breakdown, x -> x.key = 'player')[0].value AS INT) AS player_id,
        CAST(FILTER(breakdown, x -> x.key = 'team')[0].value   AS INT) AS team_id,
        FILTER(breakdown, x -> x.key = 'position')[0].value     AS position,
        metric_key,
        metric_value,
        _ingestion_timestamp
    FROM metric_rows
""")
add_pk(f"{S}.silver_season_metrics", "silver_season_metrics_pk", ["season_metric_sk"])
print("silver_season_metrics:", spark.table(f"{S}.silver_season_metrics").count())

## Verify

In [ ]:
print("=" * 60)
print("SILVER LAYER SUMMARY")
print("=" * 60)
silver_tables = spark.sql(f"""
    SELECT table_name FROM {UC_CATALOG}.information_schema.tables
    WHERE table_schema = '{SILVER_SCHEMA}' AND table_name LIKE 'silver_%'
    ORDER BY table_name
""").collect()
for r in silver_tables:
    table = f"{S}.{r['table_name']}"
    try:
        n = spark.table(table).count()
        print(f"  {table:<70s}  {n:>10,} rows")
    except Exception as e:
        print(f"  {table:<70s}  FAILED: {e}")